# VW01 2026-05-22 Experimental Procedure QC

This notebook checks whether the sensory-motor closed-loop experiment has the expected session/block structure, UUID timing, stimulus durations, wheel-to-phase coupling, and voltage channel health. It intentionally does not depend on dF/F quality.

## What Data Are Collected?

| Data | What it records | Primary use |
|---|---|---|
| Performed stimulus table | The visual command emitted by Bonsai: block, trial type, orientation, phase, duration, and UUID | Reconstruct the actual stimulus sequence and experimental conditions |
| Bonsai event logger | Timestamped display frames, `StimStart-UUID`, `StimEnd-UUID`, wheel angle, photodiode state, and session START/END markers | Establish stimulus timing and match each stimulus command to logger time |
| Prairie voltage recording | Continuously sampled physical inputs: imaging exposure, encoder phases A/B, photodiode, camera, HiFi, and optional triggers | Independently validate acquisition timing and physical wheel activity |
| Prairie acquisition XML | Microscope timing and acquisition metadata | Relate scope-exposure pulses to the expected imaging frame period and acquisition duration |
| Processed voltage HDF5 | Standardized voltage channels generated from the Prairie voltage CSV | Efficient voltage-channel health and alignment checks |
| Suite2p metadata | Registered imaging frame count, frame rate, channels, motion offsets, and processing settings | Check consistency between imaging and voltage clocks |
| dF/F and ROI outputs | Fluorescence traces and ROI masks after imaging processing | Neural response analysis; not required for procedure/timing QC |


## Which Files Contain Each Data Type?

| File or path | Format | Semantic meaning |
|---|---|---|
| `orientations_orientations0.csv` | One row per Bonsai stimulus command | Each row describes what Bonsai requested the display to show. In Block 2, a `standard` row is normally one frame-level phase update, not one analysis trial. Oddball rows are event-like presentations. `Id` is the UUID used to link the row to logger timing. |
| `orientations_logger.csv` | Long event table with `Frame`, `Timestamp`, and string `Value` | Records when Bonsai events occurred. `StimStart-<UUID>` and `StimEnd-<UUID>` delimit a command. `Wheel-Deg-*` records cumulative wheel angle. `Frame` rows track display updates. Logger photodiode rows are software-observed states, not necessarily the physical Prairie photodiode input. |
| `*VoltageRecording_001.csv` | Samples at approximately 5 kHz; one time column and eight physical input columns | Physical acquisition signals. For this setup, Input 3 is scope exposure, Inputs 6/7 are encoder quadrature A/B, and Input 1 is the physical photodiode. Channel assignments must be verified for each setup/session. |
| Main Prairie `*.xml` | Prairie acquisition metadata | Frame period and microscope acquisition configuration used to interpret the scope-exposure voltage signal. |
| `raw_voltages.h5` | HDF5 arrays under `raw/*` | Pipeline-standardized forms of the physical voltage channels and voltage timebase. |
| `suite2p/plane0/ops.npy` | Pickled Python dictionary | Suite2p frame count/rate, channel assignment, registration values, mean images, and processing parameters. |
| `dff.h5` | HDF5, typically ROI x imaging frame | Neural dF/F traces. This notebook treats these as optional because experimental procedure QC should be independent of ROI quality. |


## Important Format Distinctions

1. **A Bonsai CSV row is not always an analysis trial.** In the closed-loop mismatch block, normal 0-degree grating phase is updated frame by frame from wheel movement. Thousands of `standard` rows therefore describe an ongoing stimulus. Contiguous standard rows must be grouped into standard sequences/presentations for trial-level analysis.
2. **UUID pairing establishes command timing.** The `Id` in the performed stimulus table should have a corresponding `StimStart-Id` and `StimEnd-Id` in the logger. This validates when each command was active, but it does not by itself prove that the monitor physically displayed it.
3. **Logger time and Prairie voltage time are separate clocks.** They should agree at identifiable markers and over session duration, but alignment must be tested rather than assumed.
4. **Wheel angle and displayed phase are different units.** Bonsai scales wheel displacement before wrapping visual phase modulo `2*pi`. Correct coupling is tested from local changes and correlation, not by assuming `Phase = radians(Wheel-Deg)`.
5. **Software photodiode events and the physical photodiode voltage are different measurements.** A valid logger photodiode stream cannot replace a missing or flat physical photodiode channel when hardware display validation is required.

## Example Records From Experimental Output Files

The next cells show representative records from the files that describe the experimental procedure and its timing. The emphasis is on Bonsai stimulus commands, Bonsai event timing, wheel records, and Prairie voltage signals. XML is not row-based, so its examples show configuration elements. Raw imaging TIFFs are intentionally omitted.

In [ ]:
from pathlib import Path
from xml.etree import ElementTree as ET
import pandas as pd

EXAMPLE_RAW_SESSION = Path('/storage/project/r-fnajafi3-0/shared/2P_Imaging/VW01/VW01_20260522_ClosedLoop-1567')
EXAMPLE_BONSAI_DIR = EXAMPLE_RAW_SESSION / 'VW01_20260522T181733'
EXAMPLE_STIM_CSV = EXAMPLE_BONSAI_DIR / 'orientations_orientations0.csv'
EXAMPLE_LOGGER_CSV = EXAMPLE_BONSAI_DIR / 'orientations_logger.csv'
EXAMPLE_VOLTAGE_CSV = EXAMPLE_RAW_SESSION / 'VW01_20260522_ClosedLoop-1567_Cycle00001_VoltageRecording_001.csv'
EXAMPLE_PRAIRIE_XML = EXAMPLE_RAW_SESSION / 'VW01_20260522_ClosedLoop-1567.xml'
EXAMPLE_VOLTAGE_XML = EXAMPLE_RAW_SESSION / 'VW01_20260522_ClosedLoop-1567_Cycle00001_VoltageRecording_001.xml'

print('Performed stimulus CSV: one standard frame-update row')
stim_example = pd.read_csv(EXAMPLE_STIM_CSV, nrows=1)
display(stim_example)

print('Semantic interpretation:')
print('- Id is the UUID linking this command to StimStart/StimEnd logger events.')
print('- TrialType=standard in Block 2 means one closed-loop display update, not one analysis trial.')
print('- Orientation and Phase are in radians; Duration is in seconds.')

In [ ]:
print('Bonsai logger CSV: representative event rows')
logger_sample = pd.read_csv(EXAMPLE_LOGGER_CSV, nrows=5000)
logger_values = logger_sample['Value'].astype(str)

sample_rows = []
for label, mask in [
    ('Display frame', logger_values.eq('Frame')),
    ('Stimulus start', logger_values.str.startswith('StimStart-', na=False)),
    ('Stimulus end', logger_values.str.startswith('StimEnd-', na=False)),
    ('Wheel angle', logger_values.str.startswith('Wheel-Deg-', na=False)),
    ('Logger photodiode', logger_values.str.startswith('Photodiode-', na=False)),
]:
    match = logger_sample.loc[mask].head(1).copy()
    if not match.empty:
        match.insert(0, 'Example_Type', label)
        sample_rows.append(match)

logger_examples = pd.concat(sample_rows, ignore_index=True)
display(logger_examples)

print('Semantic interpretation: Frame is the Bonsai display-frame index; Timestamp is seconds on the Bonsai clock; Value encodes the event.')

In [ ]:
print('Prairie VoltageRecording CSV: one physical acquisition sample')
voltage_example = pd.read_csv(EXAMPLE_VOLTAGE_CSV, nrows=1)
display(voltage_example)

print('Semantic interpretation for this setup:')
print('- Time(ms): Prairie voltage-acquisition time in milliseconds.')
print('- Input 1: physical photodiode.')
print('- Input 3: microscope scope-exposure/frame TTL.')
print('- Input 6 and Input 7: wheel encoder quadrature phases A and B.')
print('- Other input assignments are summarized later and should be checked against the wiring configuration.')

In [ ]:
print('Prairie acquisition XML: timing settings')
tree = ET.parse(EXAMPLE_PRAIRIE_XML)
root = tree.getroot()

state_values = {}
for element in root.iter('PVStateValue'):
    key = element.attrib.get('key')
    if key in {'framePeriod', 'linesPerFrame', 'pixelsPerLine', 'objectiveLens', 'opticalZoom'} and key not in state_values:
        state_values[key] = element.attrib.get('value')

display(pd.DataFrame([state_values]))

print('Semantic interpretation: these settings provide the expected microscope frame timing used to interpret scope-exposure pulses in the voltage recording.')

In [ ]:
print('Prairie voltage XML: one configuration element')
voltage_tree = ET.parse(EXAMPLE_VOLTAGE_XML)
voltage_root = voltage_tree.getroot()
voltage_records = []
for element in voltage_root.iter():
    if element.attrib:
        voltage_records.append({'tag': element.tag, **element.attrib})
    if len(voltage_records) >= 5:
        break
display(pd.DataFrame(voltage_records))

print('Semantic interpretation: this XML stores voltage-recording configuration rather than sampled voltage values.')

## Main Analysis Assumptions And Tests

| Assumption | Why it matters | How this notebook tests it |
|---|---|---|
| The performed stimulus table represents what Bonsai actually issued | Planned task files may differ from behavior-dependent closed-loop output | Inventory blocks/trial types and use the performed Bonsai table as the stimulus source |
| Every relevant stimulus command has valid timing | Event-aligned analysis requires an onset and offset | Require unique UUIDs and matching `StimStart`/`StimEnd` rows; report missing/extra IDs by block |
| Logger durations agree with command durations | Incorrect start/end pairing would shift trial windows | Compare `StimEnd - StimStart` with the row's declared `Duration`, allowing display-frame timing tolerance |
| Block order and condition contents match the experimental design | Block 1 vs Block 2 comparisons are only valid if both contain the intended controls/oddballs | Check block number/type, required trial types, orientations, sequence fields, and counts |
| Block 2 standard rows are frame updates rather than independent trials | Counting each update as a trial would create tens of thousands of false trials | Collapse contiguous `standard` rows and report presentation-level condition counts |
| Wheel motion controls Block 2 visual phase | This is the defining closed-loop manipulation | Align `Wheel-Deg` to standard-row starts and fit local phase change versus wheel change; require strong correlation and small residuals |
| Encoder voltage records real wheel movement | Running-gated analyses need a physical movement signal | Verify active quadrature channels A/B and compare their cumulative movement with the Bonsai wheel trajectory |
| Imaging exposure timing is stable and covers the experiment | Future neural alignment requires a reliable imaging clock | Count scope-exposure edges, estimate frame rate, and compare with Prairie XML/Suite2p frame metadata |
| Physical stimulus delivery can be independently verified | Software commands do not prove monitor output | Inspect the physical photodiode channel. If it lacks repeated transitions, mark per-presentation hardware validation unavailable rather than silently substituting logger events |
| Clocks do not drift substantially across the session | A fixed offset is insufficient if clocks diverge | Compare START/END markers, session spans, and event-to-frame residuals across time |
| Missing data are confined to non-core blocks | Missing timing in Block 1 or 2 could invalidate the main comparison | Summarize missing starts/ends per block and identify the exact affected rows |


## Interpretation Rule

The experimental procedure is considered analysis-ready only when the relevant block structure, UUID timing, durations, and wheel/phase coupling pass independently of neural quality. Imaging/ROI quality determines whether neural conclusions can be drawn, but it should not be used to excuse or infer missing behavioral and stimulus validation.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import h5py
except ImportError:
    h5py = None

plt.rcParams.update({
    'axes.grid': False,
    'figure.dpi': 120,
    'savefig.bbox': 'tight',
})

RAW_SESSION = Path('/storage/project/r-fnajafi3-0/shared/2P_Imaging/VW01/VW01_20260522_ClosedLoop-1567')
BONSAI_DIR = RAW_SESSION / 'VW01_20260522T181733'
STIM_CSV = BONSAI_DIR / 'orientations_orientations0.csv'
LOGGER_CSV = BONSAI_DIR / 'orientations_logger.csv'
VOLTAGE_CSV = RAW_SESSION / 'VW01_20260522_ClosedLoop-1567_Cycle00001_VoltageRecording_001.csv'
PROCESSED_DIR = Path('/storage/scratch1/3/grubin6/2p_processing_results/VW01_20260522_ClosedLoop-1567_xml_referenced')
RAW_VOLTAGES_H5 = PROCESSED_DIR / 'raw_voltages.h5'
OPS_NPY = PROCESSED_DIR / 'suite2p/plane0/ops.npy'

for path in [STIM_CSV, LOGGER_CSV, VOLTAGE_CSV]:
    if not path.exists():
        raise FileNotFoundError(path)

print('Stim CSV:', STIM_CSV)
print('Logger CSV:', LOGGER_CSV)
print('Voltage CSV:', VOLTAGE_CSV)
print('Processed voltage H5:', RAW_VOLTAGES_H5, RAW_VOLTAGES_H5.exists())

## Load Bonsai Stimulus And Logger Tables

In [ ]:
stim = pd.read_csv(STIM_CSV)
logger = pd.read_csv(LOGGER_CSV)

stim['Orientation_Deg'] = np.rad2deg(stim['Orientation']).round(6)
logger_value = logger['Value'].astype(str)

stim_starts = logger[logger_value.str.startswith('StimStart-', na=False)].copy()
stim_starts['Id'] = stim_starts['Value'].str.replace('StimStart-', '', regex=False)
stim_starts = stim_starts[['Id', 'Frame', 'Timestamp']].rename(columns={'Frame': 'StimStart_Frame', 'Timestamp': 'StimStart_sec'})

stim_ends = logger[logger_value.str.startswith('StimEnd-', na=False)].copy()
stim_ends['Id'] = stim_ends['Value'].str.replace('StimEnd-', '', regex=False)
stim_ends = stim_ends[['Id', 'Frame', 'Timestamp']].rename(columns={'Frame': 'StimEnd_Frame', 'Timestamp': 'StimEnd_sec'})

events = stim.merge(stim_starts, on='Id', how='left').merge(stim_ends, on='Id', how='left')
events['Logger_Duration_sec'] = events['StimEnd_sec'] - events['StimStart_sec']
events['Duration_Error_sec'] = events['Logger_Duration_sec'] - events['Duration']

print('Stim rows:', len(stim))
print('Stim unique Ids:', stim['Id'].nunique())
print('Logger StimStart rows:', len(stim_starts), 'unique:', stim_starts['Id'].nunique())
print('Logger StimEnd rows:', len(stim_ends), 'unique:', stim_ends['Id'].nunique())
display(events.head())

## What One Trial Looks Like In The Raw CSV Files

The two CSV files have different roles. `orientations_orientations0.csv` describes **what Bonsai commanded**: block, trial type, orientation, phase, duration, and UUID. `orientations_logger.csv` records **when events occurred**: `StimStart-UUID`, `StimEnd-UUID`, display-frame updates, wheel updates, and other logger events. Joining them by UUID creates a timed stimulus table.

### Static presentations outside the wheel-coupled block

In Blocks 1 and 3, the wheel does not control the displayed grating phase. One performed-stimulus CSV row normally represents one trial. Its orientation and phase remain fixed from the matching `StimStart` until `StimEnd`, even though several display frames can occur during that interval. For example, a start at logger frame 75 and an end at frame 80 means that one static stimulus remained active across roughly five display updates. It does **not** mean five trials or five phase changes. Wheel movement may still be recorded, but it does not update the stimulus in these blocks.

### Wheel-coupled standards in Block 2

In Block 2, wheel displacement controls the visual phase of the standard 0-degree grating. Bonsai therefore writes a new short `TrialType=standard` row, UUID, and start/end pair for approximately every 33 ms phase-update command. Orientation normally remains 0 degrees while `Phase` changes with wheel movement. Consecutive standard rows belong to one continuous standard bout rather than separate analysis trials. The bout begins at the first standard row's `StimStart` and ends at the final consecutive standard row's `StimEnd`. A non-standard `motor_*` row, such as an orientation oddball, halt, or omission, terminates the standard bout; the end of Block 2 can terminate the final bout.

A Block 2 oddball itself normally uses one performed-stimulus row and one UUID start/end pair. Therefore, `TrialType` comes directly from the Bonsai output, but the final analysis-level trial boundaries also depend on block structure, row order, and UUID timing.

In [ ]:
trial_columns = [
    'BlockNumber', 'BlockLabel', 'TrialNumber', 'TrialType', 'Id',
    'Duration', 'Orientation_Deg', 'Phase',
    'StimStart_sec', 'StimEnd_sec', 'Logger_Duration_sec',
]

def display_logger_boundaries(rows, label):
    ids = rows['Id'].astype(str).tolist()
    wanted = {f'StimStart-{value}' for value in ids} | {f'StimEnd-{value}' for value in ids}
    boundaries = logger.loc[logger['Value'].isin(wanted), ['Frame', 'Timestamp', 'Value']].sort_values('Timestamp')
    print(f'{label}: matching rows from orientations_logger.csv')
    display(boundaries.reset_index(drop=True))

# Typical trial outside Block 2: one performed-stimulus row and one UUID boundary pair.
other_trial = events.loc[(events['BlockNumber'] != 2) & events['StimStart_sec'].notna() & events['StimEnd_sec'].notna()].head(1)
print('EXAMPLE A: one trial outside Block 2')
print('orientations_orientations0.csv row:')
display(other_trial[trial_columns].reset_index(drop=True))
display_logger_boundaries(other_trial, 'Outside-Block-2 trial')

# Block 2 oddball: also represented by one performed-stimulus row and one UUID pair.
block2_oddball = events.loc[(events['BlockNumber'] == 2) & (events['TrialType'] != 'standard') & events['StimStart_sec'].notna() & events['StimEnd_sec'].notna()].head(1)
print('\n\nEXAMPLE B: one Block 2 oddball trial')
print('orientations_orientations0.csv row:')
display(block2_oddball[trial_columns].reset_index(drop=True))
display_logger_boundaries(block2_oddball, 'Block 2 oddball trial')

# Block 2 standard: group the first contiguous standard run into one analysis trial.
block2_ordered = events.loc[events['BlockNumber'] == 2].sort_values('TrialNumber').reset_index(drop=True)
standard_positions = np.flatnonzero(block2_ordered['TrialType'].eq('standard').to_numpy())
start_pos = int(standard_positions[0])
end_pos = start_pos
while end_pos + 1 < len(block2_ordered) and block2_ordered.iloc[end_pos + 1]['TrialType'] == 'standard':
    end_pos += 1
standard_trial = block2_ordered.iloc[start_pos:end_pos + 1]

print('\n\nEXAMPLE C: one grouped Block 2 standard trial')
standard_summary = pd.DataFrame([{
    'BlockNumber': 2,
    'Analysis_Trial_Type': 'grouped standard presentation',
    'Source_Update_Rows': len(standard_trial),
    'TrialNumber_Start': int(standard_trial['TrialNumber'].iloc[0]),
    'TrialNumber_End': int(standard_trial['TrialNumber'].iloc[-1]),
    'StimStart_sec': standard_trial['StimStart_sec'].iloc[0],
    'StimEnd_sec': standard_trial['StimEnd_sec'].iloc[-1],
    'Grouped_Duration_sec': standard_trial['StimEnd_sec'].iloc[-1] - standard_trial['StimStart_sec'].iloc[0],
}])
print('Grouped trial summary:')
display(standard_summary)
next_event = block2_ordered.iloc[end_pos + 1:end_pos + 2]
boundary_rows = pd.concat([
    standard_trial.head(1).assign(Boundary_Meaning='Standard bout begins'),
    standard_trial.tail(1).assign(Boundary_Meaning='Final standard phase update'),
    next_event.assign(Boundary_Meaning='Event terminating the standard bout'),
], ignore_index=True)
boundary_columns = ['Boundary_Meaning'] + trial_columns
print('\nPerformed-stimulus rows defining the standard bout boundary:')
display(boundary_rows[boundary_columns])

first_standard = standard_trial.head(1)
last_standard = standard_trial.tail(1)
start_value = f"StimStart-{first_standard['Id'].iloc[0]}"
end_value = f"StimEnd-{last_standard['Id'].iloc[0]}"
boundary_logger = logger.loc[
    logger['Value'].isin([start_value, end_value]),
    ['Frame', 'Timestamp', 'Value'],
].copy()
boundary_logger['Boundary_Meaning'] = boundary_logger['Value'].map({
    start_value: 'Analysis-level standard bout start',
    end_value: 'Analysis-level standard bout end',
})
print('\nLogger rows used as the grouped standard bout start and end:')
display(boundary_logger[['Boundary_Meaning', 'Frame', 'Timestamp', 'Value']].sort_values('Timestamp').reset_index(drop=True))

if next_event.empty:
    termination = 'the end of Block 2'
else:
    termination = f"the next {next_event['TrialType'].iloc[0]} event"
print(f'Interpretation: intermediate standard UUID pairs are phase updates within this bout. The bout ends immediately before {termination}.')

## Whole-Session Checks

In [ ]:
def status(ok, warn=False):
    if ok:
        return 'OK'
    return 'WARN' if warn else 'FAIL'

stim_ids = set(stim['Id'])
start_ids = set(stim_starts['Id'])
end_ids = set(stim_ends['Id'])

session_checks = pd.DataFrame([
    {
        'check': 'Stimulus Ids are unique',
        'status': status(stim['Id'].nunique() == len(stim)),
        'value': f"unique={stim['Id'].nunique()}, rows={len(stim)}",
    },
    {
        'check': 'Every stimulus row has a StimStart logger row',
        'status': status(len(stim_ids - start_ids) == 0),
        'value': f"missing={len(stim_ids - start_ids)}, extra={len(start_ids - stim_ids)}",
    },
    {
        'check': 'Every stimulus row has a StimEnd logger row',
        'status': status(len(stim_ids - end_ids) == 0, warn=True),
        'value': f"missing={len(stim_ids - end_ids)}, extra={len(end_ids - stim_ids)}",
    },
    {
        'check': 'StimStart timestamps are monotonic by TrialNumber',
        'status': status(events.sort_values('TrialNumber')['StimStart_sec'].dropna().is_monotonic_increasing),
        'value': f"non-null starts={events['StimStart_sec'].notna().sum()}",
    },
    {
        'check': 'Core Block 1 and Block 2 are present',
        'status': status({1, 2}.issubset(set(stim['BlockNumber']))),
        'value': f"blocks={sorted(stim['BlockNumber'].unique().tolist())}",
    },
])

display(session_checks)

if stim_ids - end_ids:
    print('Rows missing StimEnd:')
    display(stim[stim['Id'].isin(stim_ids - end_ids)])
if start_ids - stim_ids:
    print('Logger StimStart Ids not present in stimulus table:')
    display(pd.DataFrame({'Id': sorted(start_ids - stim_ids)}))

## Block Inventory

In [ ]:
block_summary = (
    events.groupby(['BlockNumber', 'BlockLabel', 'BlockType'], dropna=False)
    .agg(
        rows=('Id', 'size'),
        trial_start=('TrialNumber', 'min'),
        trial_end=('TrialNumber', 'max'),
        csv_duration_sec=('Duration', 'sum'),
        logger_start_sec=('StimStart_sec', 'min'),
        logger_end_sec=('StimEnd_sec', 'max'),
        missing_starts=('StimStart_sec', lambda x: int(x.isna().sum())),
        missing_ends=('StimEnd_sec', lambda x: int(x.isna().sum())),
        unique_ids=('Id', 'nunique'),
    )
    .reset_index()
)
block_summary['logger_span_sec'] = block_summary['logger_end_sec'] - block_summary['logger_start_sec']
block_summary['csv_duration_min'] = block_summary['csv_duration_sec'] / 60
block_summary['logger_span_min'] = block_summary['logger_span_sec'] / 60
display(block_summary)

trial_type_counts = (
    events.groupby(['BlockNumber', 'BlockLabel', 'BlockType', 'TrialType'], dropna=False)
    .size().rename('rows').reset_index()
)
display(trial_type_counts)

orientation_counts = (
    events.groupby(['BlockNumber', 'TrialType', 'Orientation_Deg'], dropna=False)
    .size().rename('rows').reset_index()
)
display(orientation_counts)

## Per-Block Procedure Checks

In [ ]:
expected_by_block = {
    1: {'BlockType': 'standard_control', 'required_trial_types': {'single', 'halt', 'omission'}, 'required_orientations_deg': set(np.arange(0, 315, 22.5))},
    2: {'BlockType': 'motor_oddball', 'required_trial_types': {'standard', 'motor_halt', 'motor_omission', 'motor_orientation_45', 'motor_orientation_90'}, 'required_orientations_deg': {0.0, 45.0, 90.0}},
    3: {'BlockType': 'standard_control', 'required_trial_types': {'single', 'halt', 'omission'}, 'required_orientations_deg': set(np.arange(0, 315, 22.5))},
    4: {'BlockType': 'sequential_control_block', 'required_trial_types': {'single', 'halt', 'omission'}, 'required_orientations_deg': set(np.arange(0, 315, 22.5))},
    5: {'BlockType': 'jitter_control', 'required_trial_types': {'single'}, 'required_orientations_deg': {0.0}},
    6: {'BlockType': 'open_loop_prerecorded', 'required_trial_types': {'prerecorded', 'motor_halt', 'motor_omission', 'motor_orientation_45', 'motor_orientation_90'}, 'required_orientations_deg': {0.0, 45.0, 90.0}},
    7: {'BlockType': 'movie', 'required_trial_types': {'single'}, 'required_orientations_deg': {0.0}},
}

rows = []
for block_number, spec in expected_by_block.items():
    block = events[events['BlockNumber'] == block_number]
    present = len(block) > 0
    actual_block_types = set(block['BlockType'].dropna())
    actual_trial_types = set(block['TrialType'].dropna())
    actual_orientations = set(np.round(block['Orientation_Deg'].dropna(), 6))
    expected_orientations = set(np.round(list(spec['required_orientations_deg']), 6))
    rows.extend([
        {'Block': block_number, 'check': 'block exists', 'status': status(present), 'value': len(block)},
        {'Block': block_number, 'check': 'block type matches expected', 'status': status(spec['BlockType'] in actual_block_types), 'value': sorted(actual_block_types)},
        {'Block': block_number, 'check': 'required trial types present', 'status': status(spec['required_trial_types'].issubset(actual_trial_types)), 'value': f"missing={sorted(spec['required_trial_types'] - actual_trial_types)}"},
        {'Block': block_number, 'check': 'required orientations present', 'status': status(expected_orientations.issubset(actual_orientations)), 'value': f"missing={sorted(expected_orientations - actual_orientations)}"},
        {'Block': block_number, 'check': 'all rows have StimStart', 'status': status(block['StimStart_sec'].notna().all()), 'value': int(block['StimStart_sec'].isna().sum())},
        {'Block': block_number, 'check': 'all rows have StimEnd', 'status': status(block['StimEnd_sec'].notna().all(), warn=(block_number == 7)), 'value': int(block['StimEnd_sec'].isna().sum())},
    ])

per_block_checks = pd.DataFrame(rows)
display(per_block_checks)

## Duration Checks

In [ ]:
duration_summary = (
    events[events['Logger_Duration_sec'].notna()]
    .groupby(['BlockNumber', 'TrialType', 'Duration'], dropna=False)
    .agg(
        rows=('Id', 'size'),
        logger_median_sec=('Logger_Duration_sec', 'median'),
        logger_min_sec=('Logger_Duration_sec', 'min'),
        logger_max_sec=('Logger_Duration_sec', 'max'),
        median_error_sec=('Duration_Error_sec', 'median'),
        max_abs_error_sec=('Duration_Error_sec', lambda x: float(np.nanmax(np.abs(x)))),
    )
    .reset_index()
)
duration_summary['status'] = np.where(duration_summary['max_abs_error_sec'] <= 0.05, 'OK', 'WARN')
display(duration_summary)

## Wheel Log And Phase Coupling

In [ ]:
wheel = logger[logger_value.str.startswith('Wheel-Deg-', na=False)].copy()
wheel['wheel_deg'] = wheel['Value'].str.extract(r'Wheel-Deg-\s*([-+0-9.eE]+)')[0].astype(float)
wheel = wheel.sort_values('Timestamp')

wheel_checks = pd.DataFrame([
    {'check': 'Wheel-Deg logger rows exist', 'status': status(len(wheel) > 0), 'value': len(wheel)},
    {'check': 'Wheel-Deg spans Block 2', 'status': status(len(wheel) > 0 and wheel['Timestamp'].max() >= events.loc[events['BlockNumber'].eq(2), 'StimEnd_sec'].max()), 'value': f"wheel {wheel['Timestamp'].min():.3f}-{wheel['Timestamp'].max():.3f} sec" if len(wheel) else 'none'},
    {'check': 'Wheel-Deg changes during session', 'status': status(len(wheel) > 0 and wheel['wheel_deg'].max() > wheel['wheel_deg'].min()), 'value': f"{wheel['wheel_deg'].min():.3f}-{wheel['wheel_deg'].max():.3f} deg" if len(wheel) else 'none'},
])
display(wheel_checks)

def local_phase_wheel_fit(block_number):
    block = events[(events['BlockNumber'] == block_number) & (events['TrialType'] == 'standard') & events['StimStart_sec'].notna()].copy()
    if len(block) < 10 or len(wheel) < 10:
        return None
    block = block.sort_values('StimStart_sec')
    w = np.interp(block['StimStart_sec'].to_numpy(), wheel['Timestamp'].to_numpy(), wheel['wheel_deg'].to_numpy())
    phase = block['Phase'].to_numpy()
    d_wheel = np.diff(w)
    d_phase_deg = np.rad2deg(np.angle(np.exp(1j * np.diff(phase))))
    mask = (d_wheel > 0.05) & (d_wheel < 20) & (np.abs(d_phase_deg) < 30)
    if mask.sum() < 10:
        return {'Block': block_number, 'standard_rows': len(block), 'moving_updates': int(mask.sum()), 'status': 'WARN'}
    a, b = np.linalg.lstsq(np.vstack([d_wheel[mask], np.ones(mask.sum())]).T, d_phase_deg[mask], rcond=None)[0]
    corr = np.corrcoef(d_wheel[mask], d_phase_deg[mask])[0, 1]
    pred = a * d_wheel[mask] + b
    resid = d_phase_deg[mask] - pred
    return {
        'Block': block_number,
        'standard_rows': len(block),
        'moving_updates': int(mask.sum()),
        'dphase_deg_per_wheel_deg': a,
        'offset_deg': b,
        'corr': corr,
        'median_abs_residual_deg': float(np.median(np.abs(resid))),
        'p95_abs_residual_deg': float(np.percentile(np.abs(resid), 95)),
        'status': 'OK' if corr > 0.9 and np.percentile(np.abs(resid), 95) < 5 else 'WARN',
    }

phase_fit = pd.DataFrame([x for x in [local_phase_wheel_fit(2)] if x is not None])
display(phase_fit)

In [ ]:
if len(wheel):
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    axes[0].plot(wheel['Timestamp'], wheel['wheel_deg'], lw=0.8, color='black', label='Wheel-Deg log')
    axes[0].set_ylabel('Wheel deg')
    axes[0].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    block2_std = events[(events['BlockNumber'] == 2) & (events['TrialType'] == 'standard')]
    axes[1].plot(block2_std['StimStart_sec'], block2_std['Phase'], lw=0.6, color='tab:blue', label='Block 2 standard phase')
    axes[1].set_ylabel('Phase (rad)')
    axes[1].set_xlabel('Session time (s)')
    axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
    for ax in axes:
        ax.grid(False)
    plt.show()

## Collapse Block 2 Standard Frame Updates Into Presentation-Level Trials

In [ ]:
block2 = events[events['BlockNumber'] == 2].sort_values('TrialNumber').reset_index(drop=True)
presentations = []
i = 0
analysis_trial = 1
while i < len(block2):
    row = block2.iloc[i]
    if row['TrialType'] == 'standard':
        j = i
        while j + 1 < len(block2) and block2.iloc[j + 1]['TrialType'] == 'standard':
            j += 1
        chunk = block2.iloc[i:j + 1]
        presentations.append({
            'Analysis_Trial': analysis_trial,
            'Analysis_Condition': 'standard_control',
            'Source_Update_Rows': len(chunk),
            'Trial_Number_Start': int(chunk['TrialNumber'].iloc[0]),
            'Trial_Number_End': int(chunk['TrialNumber'].iloc[-1]),
            'Stim_Start_sec': chunk['StimStart_sec'].iloc[0],
            'Stim_End_sec': chunk['StimEnd_sec'].iloc[-1],
            'Logger_Duration_sec': chunk['StimEnd_sec'].iloc[-1] - chunk['StimStart_sec'].iloc[0],
            'Orientation_Deg': 0.0,
        })
        i = j + 1
    else:
        condition = row['TrialType'].replace('motor_', '')
        presentations.append({
            'Analysis_Trial': analysis_trial,
            'Analysis_Condition': condition,
            'Source_Update_Rows': 1,
            'Trial_Number_Start': int(row['TrialNumber']),
            'Trial_Number_End': int(row['TrialNumber']),
            'Stim_Start_sec': row['StimStart_sec'],
            'Stim_End_sec': row['StimEnd_sec'],
            'Logger_Duration_sec': row['Logger_Duration_sec'],
            'Orientation_Deg': row['Orientation_Deg'],
        })
        i += 1
    analysis_trial += 1

block2_presentations = pd.DataFrame(presentations)
display(block2_presentations.head(20))
display(block2_presentations['Analysis_Condition'].value_counts())
display(block2_presentations.groupby('Analysis_Condition')['Source_Update_Rows'].describe())

## Voltage Channel Health

In [ ]:
def ttl_health(values, time_sec, threshold=0.5):
    x = np.asarray(values)
    high = x > threshold
    rising = np.flatnonzero((~high[:-1]) & high[1:]) + 1
    falling = np.flatnonzero(high[:-1] & (~high[1:])) + 1
    periods = np.diff(time_sec[rising]) if len(rising) > 1 else np.array([])
    return {
        'n_samples': len(x),
        'min': float(np.nanmin(x)),
        'max': float(np.nanmax(x)),
        'mean': float(np.nanmean(x)),
        'std': float(np.nanstd(x)),
        'threshold': threshold,
        'rising_edges': int(len(rising)),
        'falling_edges': int(len(falling)),
        'median_period_sec': float(np.nanmedian(periods)) if len(periods) else np.nan,
        'edge_rate_hz': float(1 / np.nanmedian(periods)) if len(periods) else np.nan,
        'high_fraction': float(np.nanmean(high)),
    }

if h5py is None:
    print('h5py is not installed in this kernel; skipping raw_voltages.h5 checks.')
elif not RAW_VOLTAGES_H5.exists():
    print('raw_voltages.h5 not found yet; skipping processed voltage checks.')
else:
    roles = {
        'raw/vol_img': 'Scope exposure / 2P frame TTL',
        'raw/vol_flir': 'FLIR/camera strobe',
        'raw/vol_hifi': 'HiFi TTL',
        'raw/vol_stim_aud': 'HiFi audio waveform',
        'raw/vol_pmt': 'Encoder phase A',
        'raw/vol_led': 'Encoder phase B',
        'raw/vol_2p_stim': '2P stim trigger optional',
        'raw/vol_stim_vis': 'Visual photodiode optional',
        'raw/vol_start': 'Start/trial trigger optional',
    }
    with h5py.File(RAW_VOLTAGES_H5, 'r') as f:
        time = np.asarray(f['raw/vol_time'])
        time_sec = time / 1000 if np.nanmax(time) > 100000 else time
        rows = []
        for key, role in roles.items():
            if key not in f:
                continue
            vals = np.asarray(f[key])
            kind = 'analog' if key == 'raw/vol_stim_aud' else 'ttl'
            row = {'channel': key, 'role': role, 'kind': kind}
            if kind == 'ttl':
                row.update(ttl_health(vals, time_sec, threshold=0.5))
            else:
                row.update({'n_samples': len(vals), 'min': float(np.nanmin(vals)), 'max': float(np.nanmax(vals)), 'mean': float(np.nanmean(vals)), 'std': float(np.nanstd(vals))})
            rows.append(row)
    voltage_health = pd.DataFrame(rows)
    display(voltage_health)

## Summary Verdict

In [ ]:
all_checks = pd.concat([
    session_checks.assign(scope='session', Block=np.nan),
    per_block_checks.assign(scope='block'),
], ignore_index=True, sort=False)

display(all_checks.groupby(['scope', 'status']).size().rename('count').reset_index())
display(all_checks[all_checks['status'] != 'OK'])

print('Interpretation notes:')
print('- Block 2 standard rows are frame updates and should be collapsed into presentation-level standard_control trials.')
print('- A missing StimEnd on Block 7 movie does not affect Block 1 vs Block 2 analysis.')
print('- Use the stimulus CSV Phase as the canonical displayed phase; use Wheel-Deg / encoder voltage for running and coupling checks.')